# Домашняя работа: Pandas — анализ данных платформы онлайн-обучения

Домашняя работа использует **другой датасет**, чтобы не повторять семинар.

## Получение данных

1. Запустите `generate_data.py`.
2. Введите номер варианта.
3. Будут созданы:
   - `learning_activity.csv`;
   - `courses.csv`.
4. Оба файла понадобятся в работе.

В домашней работе требуется не только получить числа, но и **интерпретировать результаты**.


## Датасет `learning_activity.csv`

Одна строка — прохождение студентом конкретного курса.

Основные поля:

- `student_id`;
- `course_id`;
- `start_date`;
- `last_activity`;
- `lessons_completed`;
- `study_hours`;
- `score`;
- `subscription`;
- `region`;
- `device`;
- `status`.

## Датасет `courses.csv`

Справочник курсов:

- `course_id`;
- `course_name`;
- `course_category`;
- `lessons_total`;
- `base_price`.


In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.float_format", "{:.2f}".format)

activity = pd.read_csv("learning_activity.csv")
courses = pd.read_csv("courses.csv")

activity.head()

,student_id,course_id,start_date,last_activity,lessons_completed,study_hours,score,subscription,region,device,status
0,2114,DA201,2026-07-18,2026-08-03,11.00,8.63,49.10,Разовая оплата,Урал,Desktop,active
1,2068,WEB101,2026-07-09,2026-09-03,27.00,21.25,89.31,Разовая оплата,Поволжье,Desktop,active
2,2019,PY101,2026-05-11,2026-06-15,21.00,15.76,90.02,Разовая оплата,Санкт-Петербург,Desktop,active
3,2092,WEB101,2026-02-19,2026-04-03,20.00,17.92,79.56,Разовая оплата,Москва,Desktop,active
4,2115,PY101,2026-03-14,2026-04-08,15.00,11.59,61.91,Разовая оплата,Сибирь,Mobile,active


---
# Задание 1. Аудит двух датасетов


Для `activity` и `courses` отдельно исследуйте `shape`, типы, пропуски, дубликаты, уникальные значения ключевых категориальных столбцов.

Определите, какие проблемы качества данных необходимо исправить до анализа.


In [2]:
def audit_data(data):
    print("Размер:", data.shape)
    display(pd.DataFrame({
        "Тип": data.dtypes,
        "Пропуски": data.isna().sum()
    }))
    print("Полных дубликатов:", data.duplicated().sum())
    categories = ["course_id", "course_name", "course_category",
                  "subscription", "region", "device", "status"]
    for column in categories:
        if column in data.columns:
            print(column, ":", data[column].unique())


print("Активность")
audit_data(activity)
print("Курсы")
audit_data(courses)

Активность
Размер: (245, 11)


,Тип,Пропуски
student_id,int64,0
course_id,object,0
start_date,object,0
last_activity,object,0
lessons_completed,float64,0
study_hours,float64,4
score,float64,7
subscription,object,0
region,object,4
device,object,3


Полных дубликатов: 5
course_id : ['DA201' 'WEB101' 'PY101' 'SQL201' 'ML301']
subscription : ['Разовая оплата' 'Месячная' 'Корпоративная' 'РАЗОВАЯ ОПЛАТА' 'Месячная '
 'корпоративная' ' месячная ' ' разовая оплата']
region : ['Урал' 'Поволжье' 'Санкт-Петербург' 'Москва' 'Сибирь' nan]
device : ['Desktop' 'Mobile' 'Desktop ' 'Tablet' nan 'desktop ' ' tablet' 'MOBILE']
status : ['active' 'dropped' 'completed']
Курсы
Размер: (5, 5)


,Тип,Пропуски
course_id,object,0
course_name,object,0
course_category,object,0
lessons_total,int64,0
base_price,int64,0


Полных дубликатов: 0
course_id : ['PY101' 'DA201' 'ML301' 'SQL201' 'WEB101']
course_name : ['Python: основы' 'Анализ данных' 'Основы ML' 'SQL и базы данных'
 'Web-разработка']
course_category : ['Программирование' 'Data Science' 'Базы данных']


В активности 245 строк и 11 столбцов, 5 полных дубликатов.
Есть пропуски в `score`, `study_hours`, `region`, `device`, разные написания
категорий и даты в строковом формате. Нужно проверить границы чисел и
корректность дат. Справочник содержит 5 курсов, без пропусков и дубликатов;
`course_id` уникален.


---
# Задание 2. Очистка данных


Очистите копию `activity`:

- обработайте пропуски;
- удалите полные дубликаты;
- приведите `subscription` и `device` к единому виду;
- преобразуйте даты;
- найдите некорректные значения `lessons_completed`, `study_hours`, `score`.

Обоснуйте решения по каждой группе ошибок.


In [3]:
def clean_activity(activity, courses):
    data = activity.drop_duplicates().copy()
    data["subscription"] = data["subscription"].str.strip().str.capitalize()
    data["device"] = data["device"].str.strip().str.title()

    for column in ["subscription", "device", "region"]:
        data[column] = data[column].fillna("Неизвестно")

    for column in ["start_date", "last_activity"]:
        data[column] = pd.to_datetime(
            data[column], format="%Y-%m-%d", errors="coerce"
        )

    limits = data["course_id"].map(courses.set_index("course_id")["lessons_total"])
    errors = {
        "lessons_completed": (data["lessons_completed"] < 0)
                             | (data["lessons_completed"] > limits),
        "study_hours": data["study_hours"] < 0,
        "score": (data["score"] < 0) | (data["score"] > 100)
    }
    print("Некорректные значения:")
    for column, mask in errors.items():
        print(column, ":", data.loc[mask, column].tolist())
        data.loc[mask, column] = np.nan

    for column in ["lessons_completed", "study_hours", "score"]:
        medians = data.groupby("course_id")[column].transform("median")
        data[column] = data[column].fillna(medians).fillna(data[column].median())
    data["lessons_completed"] = data["lessons_completed"].round()
    return data


cleaned = clean_activity(activity, courses)
print("Строк до очистки:", len(activity))
print("Строк после очистки:", len(cleaned))
print("Оставшиеся пропуски:")
display(cleaned.isna().sum())
print("Подписки:", cleaned["subscription"].unique())
print("Устройства:", cleaned["device"].unique())

Некорректные значения:
lessons_completed : [-3.0, 999.0]
study_hours : [-5.0]
score : [135.0, -10.0]
Строк до очистки: 245
Строк после очистки: 240
Оставшиеся пропуски:


student_id           0
course_id            0
start_date           2
last_activity        2
lessons_completed    0
study_hours          0
score                0
subscription         0
region               0
device               0
status               0
dtype: int64

Подписки: ['Разовая оплата' 'Месячная' 'Корпоративная']
Устройства: ['Desktop' 'Mobile' 'Tablet' 'Неизвестно']


Дубликаты удалены, осталось 240 регистраций. Отрицательные уроки
и часы, число уроков выше длины курса и баллы вне 0–100 заменены на пропуски.
Числовые пропуски заполнены медианой своего курса; если она недоступна — общей
медианой. Число уроков округлено до целого. Медиана меньше зависит от выбросов,
но заполнение может влиять на средние и корреляции.

Категории очищены от пробелов и приведены к одному регистру. Неизвестные
регион и устройство выделены в отдельную категорию. Четыре неверные даты
превращены в `NaT`: угадывать их нельзя, сами регистрации сохранены.


---
# Задание 3. Объединение таблиц


Объедините очищенную активность со справочником курсов через `merge()` по `course_id`.

Проверьте:

- изменилась ли длина таблицы;
- появились ли пропуски после объединения;
- есть ли строки без найденного курса.

Объясните, почему для этой задачи подходит выбранный вами тип объединения.


In [4]:
data = cleaned.merge(
    courses, on="course_id", how="left", validate="many_to_one", indicator=True
)

print("Строк до и после объединения:", len(cleaned), len(data))
print("Строк без найденного курса:", (data["_merge"] == "left_only").sum())
print("Пропуски в добавленных столбцах:")
display(data[["course_name", "course_category", "lessons_total", "base_price"]].isna().sum())
data = data.drop(columns="_merge")

Строк до и после объединения: 240 240
Строк без найденного курса: 0
Пропуски в добавленных столбцах:


course_name        0
course_category    0
lessons_total      0
base_price         0
dtype: int64

Выбрано левое объединение, чтобы сохранить все регистрации.
Проверка `many_to_one` запрещает размножение строк из-за повторного ключа
в справочнике. Осталось 240 строк, все курсы найдены, новых пропусков нет.
Четыре пропуска дат остались после очистки.


---
# Задание 4. Производные показатели


Создайте:

- `completion_rate = lessons_completed / lessons_total`;
- `activity_days = last_activity - start_date` в днях;
- `hours_per_lesson = study_hours / lessons_completed`.

Обработайте случаи деления на ноль.

Найдите наблюдения с самым высоким и самым низким `completion_rate`.


In [5]:
def add_metrics(data):
    data = data.copy()
    data["completion_rate"] = data["lessons_completed"] / data["lessons_total"].replace(0, np.nan)
    data["activity_days"] = (data["last_activity"] - data["start_date"]).dt.days
    data["activity_days"] = data["activity_days"].where(data["activity_days"] >= 0)
    data["hours_per_lesson"] = data["study_hours"] / data["lessons_completed"].replace(0, np.nan)
    data["completed"] = data["status"] == "completed"
    data["dropped"] = data["status"] == "dropped"
    return data


data = add_metrics(data)
columns = ["student_id", "course_name", "lessons_completed", "completion_rate"]
print("Максимальный процент прохождения:")
display(data.loc[data["completion_rate"] == data["completion_rate"].max(), columns])
print("Минимальный процент прохождения:")
display(data.loc[data["completion_rate"] == data["completion_rate"].min(), columns])

Максимальный процент прохождения:


,student_id,course_name,lessons_completed,completion_rate
95,2006,Основы ML,36.00,1.00
102,2016,Анализ данных,32.00,1.00


Минимальный процент прохождения:


,student_id,course_name,lessons_completed,completion_rate
103,2035,Web-разработка,1.00,0.03


Максимальное прохождение — 100%, минимальное — 3,33%.
При нулевом знаменателе результат остаётся неизвестным (`NaN`).
Для четырёх регистраций нельзя рассчитать длительность из-за неверной даты;
отрицательная длительность также считалась бы неизвестной.


---
# Задание 5. Анализ курсов и категорий


Для каждого курса и каждой категории курса рассчитайте:

- число регистраций;
- число уникальных студентов;
- средний процент прохождения;
- средний балл;
- среднее число часов;
- долю завершивших курс.

Определите курсы с наиболее высокой и наиболее низкой долей завершения. Для сравнений учитывайте только курсы с достаточным числом наблюдений.


In [6]:
def group_stats(data, column):
    return data.groupby(column).agg(
        registrations=("student_id", "size"),
        students=("student_id", "nunique"),
        completion_percent=("completion_rate", lambda values: values.mean() * 100),
        mean_score=("score", "mean"),
        mean_hours=("study_hours", "mean"),
        completed_share=("completed", "mean"),
        dropped_share=("dropped", "mean")
    )


by_course = group_stats(data, "course_name")
by_category = group_stats(data, "course_category")
display(by_course)
display(by_category)

enough = by_course[by_course["registrations"] >= 20]
print("Лучшая доля завершения (от 20 регистраций):")
display(enough[enough["completed_share"] == enough["completed_share"].max()])
print("Худшая доля завершения (от 20 регистраций):")
display(enough[enough["completed_share"] == enough["completed_share"].min()])

,registrations,students,completion_percent,mean_score,mean_hours,completed_share,dropped_share
course_name,,,,,,,
Python: основы,47,35,57.89,76.03,12.76,0.00,0.04
SQL и базы данных,37,34,54.05,69.84,13.54,0.00,0.08
Web-разработка,53,41,58.74,76.14,15.73,0.02,0.09
Анализ данных,59,47,57.57,72.91,16.23,0.03,0.08
Основы ML,44,36,62.69,74.60,19.52,0.02,0.07


,registrations,students,completion_percent,mean_score,mean_hours,completed_share,dropped_share
course_category,,,,,,,
Data Science,103,68,59.76,73.63,17.64,0.03,0.08
Базы данных,37,34,54.05,69.84,13.54,0.00,0.08
Программирование,100,66,58.34,76.09,14.34,0.01,0.07


Лучшая доля завершения (от 20 регистраций):


,registrations,students,completion_percent,mean_score,mean_hours,completed_share,dropped_share
course_name,,,,,,,
Анализ данных,59,47,57.57,72.91,16.23,0.03,0.08


Худшая доля завершения (от 20 регистраций):


,registrations,students,completion_percent,mean_score,mean_hours,completed_share,dropped_share
course_name,,,,,,,
Python: основы,47,35,57.89,76.03,12.76,0.00,0.04
SQL и базы данных,37,34,54.05,69.84,13.54,0.00,0.08


Для сравнения взят порог 20 регистраций; его проходят все пять курсов.
Наибольшая доля завершения у «Анализа данных»: 2 из 59, или 3,39%.
Минимальная — у «Python: основы» и «SQL и базы данных»: 0%.
Всего завершений четыре, поэтому рейтинг пока неустойчив.
В таблицах `completion_percent` выражен в процентах, доли статусов — от 0 до 1.
Завершение определяется статусом `completed`.


---
# Задание 6. Сегментация вовлечённости


Создайте группы студентов по `completion_rate`:

- `< 0.25` — низкая вовлечённость;
- `0.25–0.75` — средняя;
- `> 0.75` — высокая.

Сравните группы по среднему `score`, `study_hours` и `activity_days`.

Сформулируйте не менее двух выводов.


In [7]:
data["engagement"] = np.select(
    [data["completion_rate"] < 0.25,
     data["completion_rate"].between(0.25, 0.75),
     data["completion_rate"] > 0.75],
    ["Низкая", "Средняя", "Высокая"],
    default="Неизвестно"
)

engagement = data.groupby("engagement").agg(
    registrations=("student_id", "size"),
    mean_score=("score", "mean"),
    mean_hours=("study_hours", "mean"),
    mean_days=("activity_days", "mean")
).reindex(["Низкая", "Средняя", "Высокая"])
display(engagement)

,registrations,mean_score,mean_hours,mean_days
engagement,,,,
Низкая,16,55.80,5.26,14.00
Средняя,168,71.28,13.95,28.85
Высокая,56,87.67,23.62,47.49


1. При высокой вовлечённости средний балл составляет 87,67,
   при низкой — 55,80. Более полное прохождение связано с лучшими результатами.
2. Высокая вовлечённость сопровождается большим временем обучения:
   23,62 часа и 47,49 дня против 5,26 часа и 14 дней при низкой.

Группы относятся к прохождениям отдельных курсов: один студент может попасть
в разные группы на разных курсах. Границы 0,25 и 0,75 включены в среднюю группу.


---
# Задание 7. Подписки и устройства


Сравните типы подписки и устройства.

Для каждой группы рассчитайте:

- число наблюдений;
- средний `completion_rate`;
- средний `score`;
- долю `completed`;
- долю `dropped`.

Постройте `pivot_table`, где строки — `subscription`, столбцы — `device`, значение — средний `completion_rate`.


In [8]:
columns = ["registrations", "completion_percent", "mean_score",
           "completed_share", "dropped_share"]
by_subscription = group_stats(data, "subscription")[columns]
by_device = group_stats(data, "device")[columns]
display(by_subscription)
display(by_device)

display(data.pivot_table(
    index="subscription", columns="device",
    values="completion_rate", aggfunc="mean"
))

,registrations,completion_percent,mean_score,completed_share,dropped_share
subscription,,,,,
Корпоративная,47,55.30,73.63,0.00,0.09
Месячная,80,59.23,74.49,0.04,0.07
Разовая оплата,113,58.87,73.96,0.01,0.07


,registrations,completion_percent,mean_score,completed_share,dropped_share
device,,,,,
Desktop,141,57.95,73.52,0.01,0.08
Mobile,73,59.13,75.83,0.03,0.07
Tablet,23,59.67,72.74,0.04,0.09
Неизвестно,3,43.06,67.39,0.00,0.00


device,Desktop,Mobile,Tablet,Неизвестно
subscription,,,,
Корпоративная,0.55,0.58,0.40,NaN
Месячная,0.55,0.65,0.72,NaN
Разовая оплата,0.62,0.57,0.56,0.43


Среднее прохождение выше у месячной подписки: 59,23%,
ниже у корпоративной: 55,30%. На Tablet показатель равен 59,67%,
на Mobile — 59,13%, на Desktop — 57,95%. Это описательные различия:
состав групп и число наблюдений отличаются, причинный эффект не установлен.


---
# Задание 8. Динамика регистраций


Используя `start_date`, создайте месяц регистрации.

Для каждого месяца вычислите:

- число новых регистраций;
- средний балл;
- средний процент прохождения.

Определите месяц с максимальным числом регистраций.

Дополнительно постройте сводную таблицу: месяц × категория курса, значение — число регистраций.


In [9]:
data["registration_month"] = data["start_date"].dt.to_period("M")
monthly = data.groupby("registration_month").agg(
    registrations=("student_id", "size"),
    mean_score=("score", "mean"),
    completion_percent=("completion_rate", lambda values: values.mean() * 100)
)
display(monthly)
print("Месяц с максимальным числом регистраций:")
display(monthly[monthly["registrations"] == monthly["registrations"].max()])
print("Регистраций с неизвестным месяцем:", data["registration_month"].isna().sum())

display(data.pivot_table(
    index="registration_month", columns="course_category",
    values="student_id", aggfunc="count", fill_value=0
))

,registrations,mean_score,completion_percent
registration_month,,,
2026-02,40,72.63,56.80
2026-03,37,74.95,59.97
2026-04,32,73.13,54.08
2026-05,44,75.64,58.99
2026-06,49,74.29,58.51
2026-07,36,73.81,60.59


Месяц с максимальным числом регистраций:


,registrations,mean_score,completion_percent
registration_month,,,
2026-06,49,74.29,58.51


Регистраций с неизвестным месяцем: 2


course_category,Data Science,Базы данных,Программирование
registration_month,,,
2026-02,15,3,22
2026-03,12,8,17
2026-04,15,6,11
2026-05,21,6,17
2026-06,27,6,16
2026-07,12,7,17


Больше всего регистраций в июне 2026 года — 49.
В помесячный анализ вошло 238 регистраций; у двух неизвестна дата начала.
Сравниваются регистрации на курсы, а не число впервые пришедших студентов.


---
# Задание 9. Связи и необычные наблюдения


Для числовых показателей `completion_rate`, `study_hours`, `score`, `activity_days`, `base_price` рассчитайте корреляционную матрицу.

Найдите:

- наиболее сильную положительную связь;
- наиболее сильную отрицательную связь;
- наблюдения с подозрительно большим `hours_per_lesson`;
- студентов с высоким процентом прохождения, но низким баллом.

Объясните, почему корреляция не является доказательством причинности.


In [10]:
def find_correlations(data):
    columns = ["completion_rate", "study_hours", "score", "activity_days", "base_price"]
    matrix = data[columns].corr()
    mask = np.triu(np.ones(matrix.shape, dtype=bool), k=1)
    pairs = matrix.where(mask).stack().sort_values().rename("correlation")
    return matrix, pairs


correlations, pairs = find_correlations(data)
display(correlations)
print("Самая сильная положительная связь:")
display(pairs[pairs > 0].tail(1))
print("Самая сильная отрицательная связь:")
display(pairs[pairs < 0].head(1))

q1, q3 = data["hours_per_lesson"].quantile([0.25, 0.75])
limit = q3 + 1.5 * (q3 - q1)
columns = ["student_id", "course_name", "completion_rate", "score",
           "study_hours", "hours_per_lesson"]
unusual_hours = data[data["hours_per_lesson"] > limit]
print(f"Подозрительно большое время на урок: больше {limit:.2f} ч")
display(unusual_hours[columns].sort_values("hours_per_lesson", ascending=False))

low_scores = data[(data["completion_rate"] > 0.75) & (data["score"] < 60)]
print("Прохождение выше 75%, балл ниже 60:")
display(low_scores[columns])

,completion_rate,study_hours,score,activity_days,base_price
completion_rate,1.00,0.80,0.73,0.74,0.07
study_hours,0.80,1.00,0.56,0.89,0.29
score,0.73,0.56,1.00,0.56,-0.02
activity_days,0.74,0.89,0.56,1.00,0.24
base_price,0.07,0.29,-0.02,0.24,1.00


Самая сильная положительная связь:


study_hours  activity_days   0.89
Name: correlation, dtype: float64

Самая сильная отрицательная связь:


score  base_price   -0.02
Name: correlation, dtype: float64

Подозрительно большое время на урок: больше 1.51 ч


,student_id,course_name,completion_rate,score,study_hours,hours_per_lesson
157,2022,SQL и базы данных,0.14,54.52,11.67,2.92
103,2035,Web-разработка,0.03,69.94,2.83,2.83
133,2022,Python: основы,0.12,35.65,5.80,1.93
199,2064,Python: основы,0.29,60.33,11.59,1.66
41,2061,SQL и базы данных,0.32,67.07,14.57,1.62
34,2059,Python: основы,0.33,64.76,12.74,1.59
107,2088,Python: основы,0.50,61.51,18.69,1.56


Прохождение выше 75%, балл ниже 60:


,student_id,course_name,completion_rate,score,study_hours,hours_per_lesson


Самая сильная положительная связь — между часами обучения
и длительностью активности: 0,893. Самая сильная отрицательная — между баллом
и ценой: −0,023; она практически отсутствует.

По правилу верхней границы `Q3 + 1,5 × IQR` найдены 7 прохождений с временем
выше 1,51 часа на урок. Это повод проверить записи, а не автоматически удалить их.
Прохождений выше 75% с баллом ниже 60 нет.

Корреляция не доказывает причинность: на показатели могут одновременно влиять
сложность курса, подготовка студента и другие факторы. Заполнение пропусков
медианами также влияет на оценки связей.


---
# Задание 10. Итоговый аналитический отчёт


Напишите функцию:

```python
def analyze_learning_platform(activity, courses):
    ...
```

Функция не должна менять исходные DataFrame.

В отчёте должны присутствовать:

- качество исходных данных;
- число студентов и регистраций;
- число курсов;
- средний процент прохождения;
- средний балл;
- доля завершивших и бросивших;
- лучший и худший курс по завершению;
- сравнение типов подписки;
- наиболее заметные корреляции.

После отчёта напишите **не менее пяти содержательных выводов**.


In [11]:
def analyze_learning_platform(activity, courses):
    print("Качество исходной активности")
    audit_data(activity)
    print("Качество справочника курсов")
    audit_data(courses)

    cleaned = clean_activity(activity, courses)
    data = cleaned.merge(courses, on="course_id", how="left", validate="many_to_one")
    data = add_metrics(data)

    print("Регистраций после очистки:", len(data))
    print("Уникальных студентов:", data["student_id"].nunique())
    print("Курсов в справочнике:", courses["course_id"].nunique())
    print("Курсов с регистрациями:", data["course_id"].nunique())
    print(f"Средний процент прохождения: {data['completion_rate'].mean():.2%}")
    print(f"Средний балл: {data['score'].mean():.2f}")
    print(f"Доля завершивших: {data['completed'].mean():.2%}")
    print(f"Доля бросивших: {data['dropped'].mean():.2%}")

    by_course = group_stats(data, "course_name")
    enough = by_course[by_course["registrations"] >= 20]
    if not enough.empty:
        print("Лучшие курсы по завершению (от 20 регистраций):")
        display(enough[enough["completed_share"] == enough["completed_share"].max()])
        print("Худшие курсы по завершению (от 20 регистраций):")
        display(enough[enough["completed_share"] == enough["completed_share"].min()])
    else:
        print("Для сравнения курсов недостаточно регистраций.")

    by_subscription = group_stats(data, "subscription")
    print("Сравнение подписок:")
    display(by_subscription)

    matrix, pairs = find_correlations(data)
    print("Корреляционная матрица:")
    display(matrix)
    print("Самая сильная положительная связь:")
    display(pairs[pairs > 0].tail(1))
    print("Самая сильная отрицательная связь:")
    display(pairs[pairs < 0].head(1))

    return {
        "data": data,
        "courses": by_course,
        "subscriptions": by_subscription,
        "correlations": matrix
    }


report = analyze_learning_platform(activity, courses)

Качество исходной активности
Размер: (245, 11)


,Тип,Пропуски
student_id,int64,0
course_id,object,0
start_date,object,0
last_activity,object,0
lessons_completed,float64,0
study_hours,float64,4
score,float64,7
subscription,object,0
region,object,4
device,object,3


Полных дубликатов: 5
course_id : ['DA201' 'WEB101' 'PY101' 'SQL201' 'ML301']
subscription : ['Разовая оплата' 'Месячная' 'Корпоративная' 'РАЗОВАЯ ОПЛАТА' 'Месячная '
 'корпоративная' ' месячная ' ' разовая оплата']
region : ['Урал' 'Поволжье' 'Санкт-Петербург' 'Москва' 'Сибирь' nan]
device : ['Desktop' 'Mobile' 'Desktop ' 'Tablet' nan 'desktop ' ' tablet' 'MOBILE']
status : ['active' 'dropped' 'completed']
Качество справочника курсов
Размер: (5, 5)


,Тип,Пропуски
course_id,object,0
course_name,object,0
course_category,object,0
lessons_total,int64,0
base_price,int64,0


Полных дубликатов: 0
course_id : ['PY101' 'DA201' 'ML301' 'SQL201' 'WEB101']
course_name : ['Python: основы' 'Анализ данных' 'Основы ML' 'SQL и базы данных'
 'Web-разработка']
course_category : ['Программирование' 'Data Science' 'Базы данных']
Некорректные значения:
lessons_completed : [-3.0, 999.0]
study_hours : [-5.0]
score : [135.0, -10.0]
Регистраций после очистки: 240
Уникальных студентов: 101
Курсов в справочнике: 5
Курсов с регистрациями: 5
Средний процент прохождения: 58.29%
Средний балл: 74.07
Доля завершивших: 1.67%
Доля бросивших: 7.50%
Лучшие курсы по завершению (от 20 регистраций):


...

,completion_rate,study_hours,score,activity_days,base_price
completion_rate,1.00,0.80,0.73,0.74,0.07
study_hours,0.80,1.00,0.56,0.89,0.29
score,0.73,0.56,1.00,0.56,-0.02
activity_days,0.74,0.89,0.56,1.00,0.24
base_price,0.07,0.29,-0.02,0.24,1.00


Самая сильная положительная связь:


study_hours  activity_days   0.89
Name: correlation, dtype: float64

Самая сильная отрицательная связь:


score  base_price   -0.02
Name: correlation, dtype: float64

Итоговые выводы:

1. После удаления 5 дубликатов осталось 240 регистраций от 101 студента
   на 5 курсов. Несколько регистраций одного студента не являются полными дубликатами.
2. Среднее прохождение — 58,29%, средний балл — 74,07.
   Завершены 4 регистрации (1,67%), брошены 18 (7,50%); большинство ещё активно.
3. По доле завершения лидирует «Анализ данных» (3,39%). У Python и SQL — 0%,
   но четырёх завершений недостаточно для устойчивой оценки качества курсов.
4. У высокой вовлечённости средний балл на 31,87 выше, чем у низкой.
   Связь заметна, но сама по себе не доказывает влияние числа пройденных уроков.
5. Месячная подписка опережает корпоративную по среднему прохождению
   на 3,93 процентного пункта. Без учёта состава групп нельзя утверждать,
   что причина различия — тип подписки.
6. Пик регистраций приходится на июнь — 49. Две записи с неверной датой
   начала исключены только из помесячных расчётов.
7. Часы обучения тесно связаны с длительностью активности (0,893).
   Цена почти не связана с баллом (−0,023). Семь записей с большим временем
   на урок требуют проверки; они сохранены в анализе.

Все доли статусов рассчитаны по регистрациям. Неизвестные даты не заполнены,
поэтому средние длительности и корреляции с ними используют доступные пары.
Функция заново очищает переданные таблицы и не изменяет исходные DataFrame.
